# Notebook description
This notebook detects potential outliers per class in the dataset, using the Isolation Forest algorithm from sklearn. The contamination parameter regulates the number of outliers to be detected.


In [ ]:
from google.colab import drive
drive.mount("/gdrive")
current_dir = "/gdrive/My\\ Drive/challengeAN2DL"
%cd $current_dir

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/My Drive/challengeAN2DL


In [ ]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [ ]:
# DATASET NAMES
DATASET_NAME_TRAINING = "an2dl2526c1/pirate_pain_train.csv"
DATASET_NAME_LABELS = "an2dl2526c1/pirate_pain_train_labels.csv"

df = pd.read_csv(DATASET_NAME_TRAINING)
labels = pd.read_csv(DATASET_NAME_LABELS)

In [ ]:

# Drop constant features
df = df.drop('joint_30', axis=1)

# Transformation to float32
# pain_survey
for index in range(4):
    i = index + 1
    column = 'pain_survey_' + str(i)
    df[column] = df[column].astype(np.float32)

# n_legs/hands/eyes
df["n_legs"] = df["n_legs"].map({"two": 2, "one+peg_leg": 1}).astype("float32")
df["n_hands"] = df["n_hands"].map({"two": 2, "one+hook_hand": 1}).astype("float32")
df["n_eyes"] = df["n_eyes"].map({"two": 2, "one+eye_patch": 1}).astype("float32")

# joints
for index in range(30):
    column = 'joint_' + str(index).zfill(2)
    df[column] = df[column].astype(np.float32)

# Class label mapping
label_mapping = {
    'no_pain': 0,
    'low_pain': 1,
    'high_pain': 2
}

labels['label'] = labels['label'].map(label_mapping)
df = df.merge(labels, on='sample_index', how='left')


In [ ]:
# Detect potential outliers with isolation forest

outlier_results = []

for label, subdf in df.groupby('label'):
    feature_cols = [c for c in df.columns if c not in ['sample_index', 'time', 'label']]

    # Compute aggregated statistics
    agg_df = subdf.groupby('sample_index')[feature_cols].agg(['mean','std','min','max','skew',pd.Series.kurt])
    agg_df.columns = [f"{col}_{func}" for col, func in agg_df.columns]

    # Scale statistics
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(agg_df)

    # Detect outliers
    iso = IsolationForest(contamination=0.05, random_state=42)
    preds = iso.fit_predict(X_scaled)

    agg_df['outlier_flag'] = preds
    agg_df['label'] = label
    outlier_results.append(agg_df)

all_outliers = pd.concat(outlier_results)
outliers_0 = all_outliers[(all_outliers['outlier_flag'] == -1) & (all_outliers['label'] == 0)]
outliers_1 = all_outliers[(all_outliers['outlier_flag'] == -1) & (all_outliers['label'] == 1)]
outliers_2 = all_outliers[(all_outliers['outlier_flag'] == -1) & (all_outliers['label'] == 2)]
print("Potential outliers for class 0: ", outliers_0.index.tolist())
print("Potential outliers for class 1: ", outliers_1.index.tolist())
print("Potential outliers for class 2: ", outliers_2.index.tolist())

Potential outliers for class 0:  [16, 21, 77, 137, 152, 160, 178, 209, 225, 264, 290, 348, 357, 381, 393, 418, 439, 444, 454, 511, 513, 521, 561, 566, 628, 657]
Potential outliers for class 1:  [44, 239, 267, 338, 465]
Potential outliers for class 2:  [88, 92, 300]
